# 03 – Baseline Models: ARIMA, Prophet, XGBoost

This notebook trains and evaluates three baseline forecasting models:
- **ARIMA** – classical statistical time-series model
- **Prophet** – Facebook/Meta time-series model with trend/seasonality decomposition
- **XGBoost** – gradient-boosted trees with engineered features

Predictions are saved for use in notebook 05 (ensemble).

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
from pathlib import Path

from src.utils.data_loader import load_processed_data
from src.utils.metrics import calculate_metrics
from src.visualization.plotter import Plotter
from src.config import RESULTS_DIR

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
plotter = Plotter()

In [2]:
# ── Load processed data ─────────────────────────────────────────────────────
train = load_processed_data('train')
val   = load_processed_data('val')
test  = load_processed_data('test')

print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

target_col = 'Close'
y_test = test[target_col]

Train: 1006 | Val: 252 | Test: 249


## A. ARIMA

In [3]:

# ── ARIMA Model ──────────────────────────────────────────────────────────
from src.models.arima_model import ARIMAModel
from src.utils.metrics import calculate_metrics
from src.visualization.plotter import Plotter

plotter = Plotter()
arima_order = (5, 1, 0)
window = 126
arima = ARIMAModel(order=arima_order)



print(f"--- Training ARIMA (Walk-forward | pdq={arima_order} | window={window} | Metrics=Validation) ---")

arima_val_preds, arima_test_preds = arima.train_and_refit(
train, val, test, target_col="Close", window=window
)

print(f"\n [Phase 1] 2024 Validation Metrics (pdq={arima_order} | window={window}):")
print(calculate_metrics(val['Close'], arima_val_preds))

print(f"\n [Phase 2] 2025 Test Metrics (pdq={arima_order} | window={window}):")
print(calculate_metrics(test['Close'], arima_test_preds))

# 参数标签：用于 title 和 filename
p, d, q = arima_order
pdq_tag = f"p{p}_d{d}_q{q}"
window_tag = f"w{window}"

# 2024 Validation 预测对比图
plotter.plot_predictions_comparison(
    y_true=val["Close"],
    predictions={"ARIMA Forecast": arima_val_preds},
    dates=val.index,
    title=f"ARIMA Validation Predictions (2024 | pdq={arima_order} | window={window})",
    filename=f"arima_val_2024_{pdq_tag}_{window_tag}.png"
)

# 2025 Test 预测对比图
plotter.plot_predictions_comparison(
    y_true=test["Close"],
    predictions={"ARIMA Forecast": arima_test_preds},
    dates=test.index,
    title=f"ARIMA Test Predictions (2025 | pdq={arima_order} | window={window})",
    filename=f"arima_test_2025_{pdq_tag}_{window_tag}.png"
)


I0000 00:00:1774925281.211809   20724 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774925299.761294   20724 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774925309.930874   20724 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


--- Training ARIMA (Walk-forward | pdq=(5, 1, 0) | window=126 | Metrics=Validation) ---


/workspaces/COMP5152ADA_Project_2/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/workspaces/COMP5152ADA_Project_2/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/workspaces/COMP5152ADA_Project_2/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/workspaces/COMP5152ADA_Project_2/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Predicti


 [Phase 1] 2024 Validation Metrics (pdq=(5, 1, 0) | window=126):
{'mse': 51232.39291645581, 'rmse': 226.34573757076984, 'mae': 165.65682981448043, 'mape': 0.8687142028760877, 'directional_accuracy': 0.545816733067729}

 [Phase 2] 2025 Test Metrics (pdq=(5, 1, 0) | window=126):
{'mse': 97395.31124426315, 'rmse': 312.08221872491094, 'mae': 216.2611427762305, 'mape': 0.9988200302470549, 'directional_accuracy': 0.47580645161290325}


## B. Prophet

In [4]:

# ── 5. Prophet Model ────────────────────────────────────────────────────────
from src.models.prophet_model import ProphetModel

prophet_cps = 0.05    # changepoint_prior_scale
prophet_cpr = 0.8 # changepoint_range
prophet_sps = 5    # seasonality_prior_scale

prophet = ProphetModel(
    changepoint_prior_scale=prophet_cps,
    changepoint_range=prophet_cpr,
    seasonality_prior_scale=prophet_sps,
)

# 参数标签：用于 title 和 filename
prophet_tag = f"cps{prophet_cps}_cpr{prophet_cpr}_sps{prophet_sps}"

print(f"--- Training Prophet (Two-Phase | {prophet_tag}) ---")
prophet_val_preds, prophet_test_preds = prophet.train_and_refit(train, val, test)

print(f"\n [Phase 1] 2024 Validation Metrics ({prophet_tag}):")
print(calculate_metrics(val['Close'], prophet_val_preds))

print(f"\n [Phase 2] 2025 Test Metrics ({prophet_tag}):")
print(calculate_metrics(test['Close'], prophet_test_preds))

plotter.plot_predictions_comparison(
    y_true=val["Close"],
    predictions={"Prophet Forecast": prophet_val_preds},
    dates=val.index,
    title=f"Prophet 2024 Validation Predictions ({prophet_tag})",
    filename=f"prophet_val_forecast_2024_{prophet_tag}.png"
)

plotter.plot_predictions_comparison(
    y_true=test['Close'],
    predictions={'Prophet Forecast': prophet_test_preds},
    dates=test.index,
    title=f"Prophet 2025 Test Predictions ({prophet_tag})",
    filename=f"prophet_test_forecast_2025_{prophet_tag}.png"
)


--- Training Prophet (Two-Phase | cps0.05_cpr0.8_sps5) ---


Importing plotly failed. Interactive plots will not work.
02:51:24 - cmdstanpy - INFO - Chain [1] start processing
02:51:25 - cmdstanpy - INFO - Chain [1] done processing
02:51:25 - cmdstanpy - INFO - Chain [1] start processing
02:51:25 - cmdstanpy - INFO - Chain [1] done processing



 [Phase 1] 2024 Validation Metrics (cps0.05_cpr0.8_sps5):
{'mse': 5617805.75543343, 'rmse': 2370.1910799413263, 'mae': 2032.4446736252492, 'mape': 10.517458516543645, 'directional_accuracy': 0.5338645418326693}

 [Phase 2] 2025 Test Metrics (cps0.05_cpr0.8_sps5):
{'mse': 4938523.438512208, 'rmse': 2222.278884053981, 'mae': 1886.122973835855, 'mape': 8.84234023028247, 'directional_accuracy': 0.5362903225806451}


## C. XGBoost

In [5]:
# ── 6. XGBoost Model (Selective Shift-1 for OHLCV + Technical Indicators+ Sentiment) ──
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(project_root))
print("cwd:", cwd)
print("project_root:", project_root)
from src.models.xgboost_model import XGBoostModel
import pandas as pd
import matplotlib.pyplot as plt

target_col = "Close"

# 候选特征：保留 Close 作为特征来源（会被 shift(1)），排除 Adj Close
base_features = [col for col in train.columns if col != "Adj Close"]

# 需要做 shift(1) 的特征集合
ohlcv_cols = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in base_features]
volume_ma_cols = [c for c in base_features if c.startswith("Volume_MA")]
ma_cols = [c for c in base_features if c.startswith("MA")]
rsi_cols = [c for c in base_features if c == "RSI"]
macd_cols = [c for c in base_features if c in ["MACD", "MACD_signal", "MACD_hist"]]
bb_cols = [c for c in base_features if c.startswith("BB_")]
obv_cols = [c for c in base_features if c == "OBV"]
sentiment_cols = [c for c in base_features if "sentiment" in c.lower()]
bool_cols = [
    c for c in base_features
    if c.startswith("is_new_high_")
    or c.startswith("is_new_low_")
    or c.startswith("above_MA")
    or c in ["ma10_above_ma20", "ma20_above_ma50"]
]

cols_to_shift = (
    ohlcv_cols
    + volume_ma_cols
    + ma_cols
    + rsi_cols
    + macd_cols
    + bb_cols
    + obv_cols
    + sentiment_cols
    + bool_cols
)
cols_to_shift = list(dict.fromkeys(cols_to_shift))  # 去重并保持顺序

print(
    f"Applying shift(1) to {len(cols_to_shift)} columns: OHLCV + Volume_MA + MA/RSI/MACD/Bollinger + OBV + Sentiment"
 )

def build_xgb_dataset(df: pd.DataFrame, all_features: list[str], shift_cols: list[str], target: str):
    X = df[all_features].copy()

    # 1) 仅指定列 shift(1) 并重命名
    X_shift = X[shift_cols].shift(1)
    X_shift.columns = [f"{c}_lag1" for c in shift_cols]

    # 2) 其余列保持原值（例如 lag_1~lag_5、宏观 Lag35、等）
    keep_cols = [c for c in all_features if c not in shift_cols and c != target]
    X_keep = X[keep_cols].copy()

    # 3) 合并成最终特征
    X_final = pd.concat([X_shift, X_keep], axis=1)

    # shift 后首行 NaN 兜底
    X_final.bfill(inplace=True)

    # 拼回目标列（目标保持原时点，不shift）
    out_df = pd.concat([X_final, df[target]], axis=1)
    feature_cols = list(X_final.columns)
    return out_df, feature_cols

train_xgb_df, xgb_feature_cols = build_xgb_dataset(train, base_features, cols_to_shift, target_col)
val_xgb_df, _ = build_xgb_dataset(val, base_features, cols_to_shift, target_col)
test_xgb_df, _ = build_xgb_dataset(test, base_features, cols_to_shift, target_col)

print(f"XGBoost uses {len(xgb_feature_cols)} features.")
print("Shifted columns are suffixed with _lag1.")

# 4. 两阶段训练
xgb = XGBoostModel()
print("--- Training XGBoost (Two-Phase Refitting) ---")
xgb_val_preds, xgb_test_preds = xgb.train_and_refit(
    train_xgb_df, val_xgb_df, test_xgb_df,
    features=xgb_feature_cols,
    target_col=target_col,
)
print(xgb.params)
print({k: xgb._model.get_xgb_params().get(k) for k in ["max_depth","learning_rate","min_child_weight","gamma","reg_alpha","reg_lambda"]})

print("\n[Phase 1] 2024 Validation Metrics:")
print(calculate_metrics(val["Close"], xgb_val_preds))

print("\n[Phase 2] 2025 Test Metrics:")
print(calculate_metrics(test["Close"], xgb_test_preds))

plotter.plot_predictions_comparison(
    y_true=val["Close"],
    predictions={"XGBoost Forecast": xgb_val_preds},
    dates=val.index,
    title="XGBoost 2024 Validation Predictions (return->close)",
    filename="xgboost_val_forecast_close_2024(return->close)_2.png",
)

plotter.plot_predictions_comparison(
    y_true=test["Close"],
    predictions={"XGBoost Forecast": xgb_test_preds},
    dates=test.index,
    title="XGBoost 2025 Test Predictions (return->close)",
    filename="xgboost_test_forecast_close_2025(return->close)_2.png",
)

# ---- Return prediction plots ----
val_close_with_prev = pd.concat([train[target_col].tail(1), val[target_col]])
val_returns_true = val_close_with_prev.pct_change().iloc[1:]
val_return_preds = pd.Series(
    xgb.predict(val_xgb_df[xgb_feature_cols]),
    index=val_xgb_df.index,
    name="XGBoost_Return",
)
val_return_preds = val_return_preds.reindex(val_returns_true.index)

val_return_metrics = calculate_metrics(
    val_returns_true,
    val_return_preds,
 )
print("\n[Return Metrics] 2024 Validation:")
print(val_return_metrics)

val_returns_true_pct = val_returns_true * 100
val_return_preds_pct = val_return_preds * 100
print("Val return preds (%):")
print(val_return_preds_pct.describe())

plotter.plot_predictions_comparison(
    y_true=val_returns_true_pct,
    predictions={"XGBoost Return": val_return_preds_pct},
    dates=val.index,
    title="XGBoost 2024 Validation Return Predictions (return->return)",
    filename="xgboost_val_forecast_return_2024(return->return)_2.png",
    y_label="Return (%)",
)

test_close_with_prev = pd.concat([val[target_col].tail(1), test[target_col]])
test_returns_true = test_close_with_prev.pct_change().iloc[1:]
test_return_preds = pd.Series(
    xgb.predict(test_xgb_df[xgb_feature_cols]),
    index=test_xgb_df.index,
    name="XGBoost_Return",
)
test_return_preds = test_return_preds.reindex(test_returns_true.index)

test_return_metrics = calculate_metrics(
    test_returns_true,
    test_return_preds,
 )
print("\n[Return Metrics] 2025 Test:")
print(test_return_metrics)

test_returns_true_pct = test_returns_true * 100
test_return_preds_pct = test_return_preds * 100
print("Test return preds (%):")
print(test_return_preds_pct.describe())

plotter.plot_predictions_comparison(
    y_true=test_returns_true_pct,
    predictions={"XGBoost Return": test_return_preds_pct},
    dates=test.index,
    title="XGBoost 2025 Test Return Predictions (return->return)",
    filename="xgboost_test_forecast_return_2025(return->return)_2.png",
    y_label="Return (%)",
)

# ---- Save return/close series to results ----
val_output = pd.DataFrame(
    {
        "actual_close": val[target_col],
        "pred_close_from_return": xgb_val_preds.reindex(val.index),
        "actual_return": val_returns_true,
        "pred_return": val_return_preds,
    }
)
val_output.to_csv(RESULTS_DIR / "xgboost_val_returns_and_close.csv")


test_output = pd.DataFrame(
    {
        "actual_close": test[target_col],
        "pred_close_from_return": xgb_test_preds.reindex(test.index),
        "actual_return": test_returns_true,
        "pred_return": test_return_preds,
    }
)
test_output.to_csv(RESULTS_DIR / "xgboost_test_returns_and_close.csv")

# ---- Phase 1-only model for feature importance (return target) ----
train_returns_p1 = train[target_col].pct_change().dropna()
train_features_p1 = train_xgb_df.loc[train_returns_p1.index, xgb_feature_cols]
val_close_with_prev_p1 = pd.concat([train[target_col].tail(1), val[target_col]])
val_returns_p1 = val_close_with_prev_p1.pct_change().iloc[1:]
val_features_p1 = val_xgb_df.loc[val_returns_p1.index, xgb_feature_cols]

xgb_phase1 = XGBoostModel()
xgb_phase1.fit(
    train_features_p1,
    train_returns_p1,
    X_val=val_features_p1,
    y_val=val_returns_p1,
)

xgb_importance_train = pd.Series(
    xgb_phase1._model.feature_importances_,
    index=xgb_feature_cols,
)

plotter.plot_feature_importance(
    xgb_importance_train,
    top_n=20,
    title="XGBoost Feature Importance (train) [return]",
    filename="xgboost_feature_importance_train_return_2020-2023_2.png",
    annotate=True,
    decimals=4,
)

# ---- Phase 2 refitted importance (return target) ----
xgb_importance_refit = pd.Series(
    xgb._model.feature_importances_,
    index=xgb_feature_cols,
)

if xgb_importance_refit.sum() == 0:
    gain_scores = xgb._model.get_booster().get_score(importance_type="gain")
    if gain_scores:
        mapped_scores = {}
        if all(k.startswith("f") for k in gain_scores):
            for k, v in gain_scores.items():
                idx = k[1:]
                if idx.isdigit():
                    i = int(idx)
                    if i < len(xgb_feature_cols):
                        mapped_scores[xgb_feature_cols[i]] = v
        else:
            mapped_scores = gain_scores
        xgb_importance_refit = pd.Series(mapped_scores).reindex(xgb_feature_cols).fillna(0)

plotter.plot_feature_importance(
    xgb_importance_refit,
    top_n=20,
    title="XGBoost Feature Importance (refit) [return]",
    filename="xgboost_feature_importance_refit_return_2024-2025_2.png",
    annotate=True,
    decimals=4,
)

cwd: /workspaces/COMP5152ADA_Project_2/notebooks
project_root: /workspaces/COMP5152ADA_Project_2
Applying shift(1) to 32 columns: OHLCV + Volume_MA + MA/RSI/MACD/Bollinger + OBV + Sentiment
XGBoost uses 41 features.
Shifted columns are suffixed with _lag1.
--- Training XGBoost (Two-Phase Refitting) ---
{'n_estimators': 1200, 'learning_rate': 0.05, 'max_depth': 4, 'objective': 'reg:absoluteerror', 'random_state': 42, 'min_child_weight': 1, 'gamma': 0, 'reg_alpha': 0, 'reg_lambda': 0.7, 'subsample': 0.85, 'colsample_bytree': 0.75, 'colsample_bylevel': 0.6, 'early_stopping_rounds': 50}
{'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 1, 'gamma': 0, 'reg_alpha': 0, 'reg_lambda': 0.7}

[Phase 1] 2024 Validation Metrics:
{'mse': 48103.50055693785, 'rmse': 219.32510243229763, 'mae': 161.70637519354054, 'mape': 0.8489727033019678, 'directional_accuracy': 0.5298804780876494}

[Phase 2] 2025 Test Metrics:
{'mse': 91504.87336583852, 'rmse': 302.49772456307585, 'mae': 204.65336335844822

In [6]:
# ---- Return metrics recap (val/test) ----
if "val_return_metrics" not in locals():
    val_close_with_prev = pd.concat([train[target_col].tail(1), val[target_col]])
    val_returns_true = val_close_with_prev.pct_change().iloc[1:]
    val_return_preds = pd.Series(
        xgb.predict(val_xgb_df[xgb_feature_cols]),
        index=val_xgb_df.index,
        name="XGBoost_Return",
    ).reindex(val_returns_true.index)
    val_return_metrics = calculate_metrics(
        val_returns_true,
        val_return_preds,
    )

if "test_return_metrics" not in locals():
    test_close_with_prev = pd.concat([val[target_col].tail(1), test[target_col]])
    test_returns_true = test_close_with_prev.pct_change().iloc[1:]
    test_return_preds = pd.Series(
        xgb.predict(test_xgb_df[xgb_feature_cols]),
        index=test_xgb_df.index,
        name="XGBoost_Return",
    ).reindex(test_returns_true.index)
    test_return_metrics = calculate_metrics(
        test_returns_true,
        test_return_preds,
    )

print("\n[Return Metrics] 2024 Validation:")
print(val_return_metrics)
print("\n[Return Metrics] 2025 Test:")
print(test_return_metrics)


[Return Metrics] 2024 Validation:
{'mse': 0.00013005511669111864, 'rmse': 0.011404171021653378, 'mae': 0.008439852063790903, 'mape': 117.0598056530046, 'directional_accuracy': 0.549800796812749}

[Return Metrics] 2025 Test:
{'mse': 0.00022191839147266368, 'rmse': 0.014896925571159429, 'mae': 0.009438129342500752, 'mape': 128.6803086365768, 'directional_accuracy': 0.4435483870967742}


## D. Comparison

In [9]:
# ── 7. Aggregate and Save Final Metrics (Val 2024 + Test 2025) ───────────
from src.utils.metrics import calculate_metrics
import pandas as pd
from src.config import RESULTS_DIR

# ── 2024 Validation Set Metrics ──────────────────────────────────────────
arima_val_metrics   = calculate_metrics(val['Close'], arima_val_preds)
prophet_val_metrics = calculate_metrics(val['Close'], prophet_val_preds)
xgb_val_metrics     = calculate_metrics(val['Close'], xgb_val_preds)

all_val_metrics = {
    'ARIMA':   arima_val_metrics,
    'Prophet': prophet_val_metrics,
    'XGBoost': xgb_val_metrics,
}

val_metrics_df = pd.DataFrame(all_val_metrics).T

print("\n BASELINE METRICS — 2024 Validation Set ")
print("=" * 70)
print(val_metrics_df.to_string())
print("=" * 70)

# ── 2025 Test Set Metrics ─────────────────────────────────────────────────
arima_metrics   = calculate_metrics(test['Close'], arima_test_preds)
prophet_metrics = calculate_metrics(test['Close'], prophet_test_preds)
xgb_metrics     = calculate_metrics(test['Close'], xgb_test_preds)

all_metrics = {
    'ARIMA':   arima_metrics,
    'Prophet': prophet_metrics,
    'XGBoost': xgb_metrics,
}

metrics_df = pd.DataFrame(all_metrics).T

print("\n FINAL BASELINE METRICS — 2025 Test Set (Refitted) ")
print("=" * 70)
print(metrics_df.to_string())
print("=" * 70)

# ── Save test metrics (primary) ───────────────────────────────────────────
metrics_df.to_csv(RESULTS_DIR / 'baseline_metrics.csv')
print(f"\n✅ Test metrics saved to {RESULTS_DIR / 'baseline_metrics.csv'}")

# 可选：绘制所有 Baseline 模型的指标对比柱状图
plotter.plot_metrics_comparison(all_metrics, filename='baseline_metrics_comparison.png')



 BASELINE METRICS — 2024 Validation Set 
                  mse         rmse          mae       mape  directional_accuracy
ARIMA    5.123239e+04   226.345738   165.656830   0.868714              0.545817
Prophet  5.617806e+06  2370.191080  2032.444674  10.517459              0.533865
XGBoost  4.810350e+04   219.325102   161.706375   0.848973              0.529880

 FINAL BASELINE METRICS — 2025 Test Set (Refitted) 
                  mse         rmse          mae      mape  directional_accuracy
ARIMA    9.739531e+04   312.082219   216.261143  0.998820              0.475806
Prophet  4.938523e+06  2222.278884  1886.122974  8.842340              0.536290
XGBoost  9.150487e+04   302.497725   204.653363  0.943323              0.463710

✅ Test metrics saved to /workspaces/COMP5152ADA_Project_2/reports/results/baseline_metrics.csv


In [8]:
# ── 8. Save Phase 1 & Phase 2 Predictions ────────────────────────────────
from src.config import RESULTS_DIR
import pandas as pd

# 1. 横向拼接各个模型的结果，分别保存 val 和 test
val_preds_df = pd.concat([arima_val_preds, prophet_val_preds, xgb_val_preds], axis=1)
test_preds_df = pd.concat([arima_test_preds, prophet_test_preds, xgb_test_preds], axis=1)

# 2. 重命名列，保持规范
val_preds_df.columns = ['ARIMA', 'Prophet', 'XGBoost']
test_preds_df.columns = ['ARIMA', 'Prophet', 'XGBoost']

# 3. 分别保存 2024 Val 和 2025 Test 预测结果
val_save_path = RESULTS_DIR / 'baseline_val_preds.csv'
test_save_path = RESULTS_DIR / 'baseline_test_preds.csv'
val_preds_df.to_csv(val_save_path)
test_preds_df.to_csv(test_save_path)

print(f"Val predictions (2024) saved to {val_save_path}  [{len(val_preds_df)} rows]")
print(f"Test predictions (2025) saved to {test_save_path}  [{len(test_preds_df)} rows]")

Val predictions (2024) saved to /workspaces/COMP5152ADA_Project_2/reports/results/baseline_val_preds.csv  [252 rows]
Test predictions (2025) saved to /workspaces/COMP5152ADA_Project_2/reports/results/baseline_test_preds.csv  [249 rows]


## Summary

See `reports/results/baseline_metrics.csv` for a full metrics table.

Continue to **04_model_lstm.ipynb** for the deep learning model.